In [1]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

nome_dados_treinamento = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos_sem_xss.csv'
nome_dados_teste = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos_com_xss.csv'

In [3]:
print("Carregando os datasets com redução de dimensionalidade...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")
display(df_treino.head())

Carregando os datasets com redução de dimensionalidade...
Dados carregados! Treino: (1979513, 52) | Teste: (848363, 52)


,Bwd Packet Length Mean,Average Packet Size,Packet Length Mean,Bwd Packet Length Std,Packet Length Variance,Init_Win_bytes_forward,Packet Length Std,Bwd Packet Length Max,Total Length of Fwd Packets,Avg Bwd Segment Size,...,Flow Duration,Fwd Packet Length Min,Down/Up Ratio,URG Flag Count,Bwd IAT Max,Active Mean,Bwd IAT Mean,Active Min,Bwd IAT Total,Label
0,0.001034,0.002825,0.002637,0.000000,1.219643e-05,0.005341,0.003493,0.000307,2.945736e-06,0.001034,...,3.229166e-05,0.000000,0.00000,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,BENIGN
1,0.000103,0.000096,0.000100,0.000109,2.232143e-08,0.018539,0.000149,0.000102,0.000000e+00,0.000103,...,2.272266e-03,0.000000,0.00641,1.0,7.645083e-04,0.0,5.603375e-04,0.0,2.241350e-03,BENIGN
2,0.000000,0.002312,0.001798,0.000000,0.000000e+00,0.078049,0.000000,0.000000,9.302326e-07,0.000000,...,1.333333e-07,0.002581,0.00000,0.0,0.000000e+00,0.0,0.000000e+00,0.0,0.000000e+00,BENIGN
3,0.010344,0.016182,0.015103,0.000000,3.428571e-06,0.000000,0.001852,0.003072,6.821705e-06,0.010344,...,1.233333e-06,0.018925,0.00641,0.0,3.333333e-08,0.0,3.333333e-08,0.0,3.333333e-08,BENIGN
4,0.019826,0.020355,0.018998,0.000000,9.905357e-05,0.000000,0.009955,0.005888,4.496124e-06,0.019826,...,1.449567e-03,0.012473,0.00641,0.0,2.500000e-08,0.0,2.500000e-08,0.0,2.500000e-08,BENIGN


,Bwd Packet Length Mean,Average Packet Size,Packet Length Mean,Bwd Packet Length Std,Packet Length Variance,Init_Win_bytes_forward,Packet Length Std,Bwd Packet Length Max,Total Length of Fwd Packets,Avg Bwd Segment Size,...,Flow Duration,Fwd Packet Length Min,Down/Up Ratio,URG Flag Count,Bwd IAT Max,Active Mean,Bwd IAT Mean,Active Min,Bwd IAT Total,Label
0,0.001034,0.001798,0.001798,0.000000,0.000000,0.003510,0.000000,0.000307,4.651163e-07,0.001034,...,9.205005e-02,0.002581,0.032051,1.0,9.166667e-02,0.000342,2.294102e-02,0.000342,9.166667e-02,BENIGN
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.003708,0.000000,0.000000,0.000000e+00,0.000000,...,1.276342e-03,0.000000,0.000000,0.0,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,BENIGN
2,0.072637,0.068038,0.077342,0.064588,0.007435,0.125015,0.086251,0.074757,1.924031e-04,0.072637,...,5.102475e-01,0.000000,0.000000,0.0,8.416667e-02,0.001497,2.998939e-02,0.000459,5.100000e-01,BENIGN
3,0.018964,0.023181,0.021635,0.000000,0.000053,0.000000,0.007293,0.005632,7.286822e-06,0.018964,...,9.841666e-06,0.020215,0.006410,0.0,4.000000e-07,0.000000,4.000000e-07,0.000000,4.000000e-07,BENIGN
4,0.000000,0.000000,0.000000,0.000000,0.000000,0.002075,0.000000,0.000000,0.000000e+00,0.000000,...,4.666666e-07,0.000000,0.006410,1.0,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000e+00,BENIGN


In [4]:
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

In [5]:
print("Codificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
num_classes = len(encoder.classes_)

Codificando as labels para a Rede Neural...


In [6]:
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [7]:
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")


Iniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...
Epoch 1/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.9468 - loss: 0.1605 - val_accuracy: 0.9655 - val_loss: 0.0897
Epoch 2/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9655 - loss: 0.0889 - val_accuracy: 0.9681 - val_loss: 0.0736
Epoch 3/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9680 - loss: 0.0778 - val_accuracy: 0.9698 - val_loss: 0.0678
Epoch 4/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9701 - loss: 0.0723 - val_accuracy: 0.9739 - val_loss: 0.0626
Epoch 5/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9715 - loss: 0.0690 - val_accuracy: 0.9747 - val_loss: 0.0607
Epoch 6/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9726 - loss: 0.0662 - val_accuracy: 0.9762 - val_loss: 0.0567
Epoch 7/10
6960/6960 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - accuracy: 0.9743 - loss: 0.0620 - val_accuracy: 0.9809 - val_loss: 0.0523
Ep

In [8]:
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

print("\n=== MATRIZ DE CONFUSÃO (DNN com Redução — Sem XSS no treino) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))


Iniciando a classificação dos dados de teste...
26512/26512 ━━━━━━━━━━━━━━━━━━━━ 12s 455us/step
Classificação concluída! Tempo de previsão: 18.8898 segundos.

=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===


,Pred_BENIGN,Pred_Bot,Pred_DDoS,Pred_DoS GoldenEye,Pred_DoS Hulk,Pred_DoS Slowhttptest,Pred_DoS slowloris,Pred_FTP-Patator,Pred_Heartbleed,Pred_Infiltration,Pred_PortScan,Pred_SSH-Patator,Pred_Web Attack – Brute Force,Pred_Web Attack – Sql Injection,Pred_Web Attack – XSS
Real_BENIGN,668637,0,62,27,4345,181,16,9,0,0,7737,0,0,0,0
Real_Bot,562,2,0,0,0,0,0,0,0,0,0,0,0,0,0
Real_DDoS,434,0,37865,8,0,0,0,0,0,0,0,0,0,0,0
Real_DoS GoldenEye,116,0,0,3043,0,2,2,0,0,0,0,0,0,0,0
Real_DoS Hulk,754,0,0,0,68513,0,0,0,0,0,0,0,0,0,0
Real_DoS Slowhttptest,20,0,0,0,0,1625,13,0,0,0,0,0,0,0,0
Real_DoS slowloris,25,0,0,1,0,46,1685,0,0,0,0,0,0,0,0
Real_FTP-Patator,55,0,0,0,3,0,0,2268,0,0,0,0,0,0,0
Real_Heartbleed,2,0,0,0,0,0,0,0,4,0,0,0,0,0,0
Real_Infiltration,10,0,0,0,0,0,0,0,0,0,0,0,0,0,0



=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===
                            precision    recall  f1-score   support

                    BENIGN       0.99      0.98      0.99    681014
                       Bot       1.00      0.00      0.01       564
                      DDoS       1.00      0.99      0.99     38307
             DoS GoldenEye       0.99      0.96      0.97      3163
                  DoS Hulk       0.94      0.99      0.96     69267
          DoS Slowhttptest       0.88      0.98      0.93      1658
             DoS slowloris       0.98      0.96      0.97      1757
               FTP-Patator       0.99      0.98      0.98      2326
                Heartbleed       1.00      0.67      0.80         6
              Infiltration       0.00      0.00      0.00        10
                  PortScan       0.86      0.98      0.91     47836
               SSH-Patator       1.00      0.49      0.66      1821
  Web Attack – Brute Force       0.81      0.05      0.

In [9]:
CLASS_ALIASES_LATEX = {'BENIGN': 'BENIGN', 'Bot': 'Bot', 'DDoS': 'DDoS', 'DoS GoldenEye': 'DoS GE', 'DoS Hulk': 'Hulk', 'DoS Slowhttptest': 'Slowhttp', 'DoS slowloris': 'Slowloris', 'FTP-Patator': 'FTP', 'Heartbleed': 'Heart', 'Infiltration': 'Infil', 'PortScan': 'PortScan', 'SSH-Patator': 'SSH', 'Web Attack - Brute Force': 'Brute', 'Web Attack - Sql Injection': 'SQL', 'Web Attack - XSS': 'XSS', 'Desconhecido': 'Desconh'}


def escape_latex(value):
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(char, char) for char in str(value))


def latex_class_name(label, short=False):
    if short:
        return escape_latex(CLASS_ALIASES_LATEX.get(label, label))
    return escape_latex(label)


def format_confusion_value(value, is_diagonal):
    value = int(value)
    if is_diagonal:
        return rf"\ok{{{value}}}"
    if value != 0:
        return rf"\err{{{value}}}"
    return "0"


def make_latex_confusion_matrix(cm_values, class_labels, caption, table_label):
    headers = [latex_class_name(label, short=True) for label in class_labels]
    rows = []
    for i, real_label in enumerate(class_labels):
        row_values = [format_confusion_value(cm_values[i][j], i == j) for j in range(len(class_labels))]
        rows.append((rf"Real\_{latex_class_name(real_label, short=True)}", row_values))

    first_col_width = max([0] + [len(row_name) for row_name, _ in rows])
    col_widths = [max(len(headers[i]), *(len(values[i]) for _, values in rows)) for i in range(len(headers))]

    def format_row(first_cell, values):
        first = first_cell.ljust(first_col_width)
        rest = " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values))
        return f"            {first} & {rest} \\\\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \scriptsize",
        r"    \resizebox{\linewidth}{!}{",
        rf"        \begin{{tabular}}{{l|{'r' * len(class_labels)}}}",
        r"            \hline",
        format_row("", headers),
        r"            \hline",
    ]
    lines.extend(format_row(row_name, row_values) for row_name, row_values in rows)
    lines.extend([
        r"            \hline",
        r"        \end{tabular}",
        r"    }",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


def format_metric(value):
    return "-" if value is None else f"{value:.2f}"


def make_latex_metrics_report(y_true_values, y_pred_values, class_labels, caption, table_label):
    report = classification_report(
        y_true_values,
        y_pred_values,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    total_support = int(sum(report[label]["support"] for label in class_labels))
    rows = []
    for label in class_labels:
        metrics = report[label]
        rows.append([
            latex_class_name(label),
            format_metric(metrics["precision"]),
            format_metric(metrics["recall"]),
            format_metric(metrics["f1-score"]),
            str(int(metrics["support"])),
        ])

    rows.extend([
        [r"\textbf{Acurácia}", "-", format_metric(report["accuracy"]), "-", str(total_support)],
        [r"\textbf{Média Macro}", format_metric(report["macro avg"]["precision"]), format_metric(report["macro avg"]["recall"]), format_metric(report["macro avg"]["f1-score"]), str(total_support)],
        [r"\textbf{Média Ponderada}", format_metric(report["weighted avg"]["precision"]), format_metric(report["weighted avg"]["recall"]), format_metric(report["weighted avg"]["f1-score"]), str(total_support)],
    ])

    headers = ["Classe", "Precisão", "Revocação", "F1-score", "Suporte"]
    col_widths = [max(len(str(row[i])) for row in [headers] + rows) for i in range(len(headers))]

    def format_row(values):
        return "        " + " & ".join(str(value).ljust(col_widths[i]) for i, value in enumerate(values)) + r" \\"

    lines = [
        r"\begin{table}[H]",
        r"    \centering",
        r"    \small",
        r"    \begin{tabular}{lrrrr}",
        r"        \hline",
        format_row(headers),
        r"        \hline",
    ]
    lines.extend(format_row(row) for row in rows[:len(class_labels)])
    lines.extend([
        r"        \hline",
        format_row(rows[-3]),
        format_row(rows[-2]),
        format_row(rows[-1]),
        r"        \hline",
        r"    \end{tabular}",
        rf"    \caption{{{escape_latex(caption)}}}",
        rf"    \label{{{table_label}}}",
        r"\end{table}",
    ])
    return "\n".join(lines)


labels_latex = labels
y_true_latex = y_teste
y_pred_latex = previsoes_dnn_labels

latex_mc = make_latex_confusion_matrix(
    cm,
    labels_latex,
    "Matriz de Confusão — DNN com Redução de Dimensionalidade (CICIDS2017, Sem XSS no treino)",
    "table:dnn_cicids_reducao_sem_xss_mc",
)
latex_metricas = make_latex_metrics_report(
    y_true_latex,
    y_pred_latex,
    labels_latex,
    "Relatório de Métricas — DNN com Redução de Dimensionalidade (CICIDS2017, Sem XSS no treino)",
    "table:dnn_cicids_reducao_sem_xss_metricas",
)

print(latex_mc)
print("\n")
print(latex_metricas)

\begin{table}[H]
    \centering
    \scriptsize
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|rrrrrrrrrrrrrrr}
            \hline
                                             & BENIGN      & Bot    & DDoS       & DoS GE    & Hulk       & Slowhttp  & Slowloris & FTP       & Heart  & Infil  & PortScan   & SSH      & Web Attack – Brute Force & Web Attack – Sql Injection & Web Attack – XSS \\
            \hline
            Real\_BENIGN                     & \ok{668637} & 0      & \err{62}   & \err{27}  & \err{4345} & \err{181} & \err{16}  & \err{9}   & 0      & 0      & \err{7737} & 0        & 0                        & 0                          & 0                \\
            Real\_Bot                        & \err{562}   & \ok{2} & 0          & 0         & 0          & 0         & 0         & 0         & 0      & 0      & 0          & 0        & 0                        & 0                          & 0                \\
            Real\_DDoS                       & \err{43